In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import ast
import re
from sklearn.utils import resample
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
df_results_phone = pd.read_csv('/content/drive/MyDrive/User Study/results_phone_encode_cat_agg.csv', index_col = 0)

In [ ]:
#Let's isolate three thresholds:
df_results_phone_selected = df_results_phone.iloc[[0, 5, 10]]
df_results_phone_selected.reset_index(drop=True, inplace=True)
df_results_phone_selected

,Threshold,Number_Columns_Train,Names_Columns_Train,Names_Columns_Final,Final_Model_Parameters,"Training ACC, F1, Precision, Recall","Validation ACC, F1, Precision, Recall","Testing ACC, F1, Precision, Recall",Global Explanations,Local Explanations,Results_Training_Before_FS,Results_Training_After_FS
0,2.89,19,"['paperlessbilling', 'paymentmethod', 'depende...","['contract', 'monthlycharges', 'totalcharges',...","{'ccp_alpha': 0.0, 'class_weight': 'balanced',...","(np.float64(0.7826846497396296), 0.64679846562...","(np.float64(0.7376510095642883), np.float64(0....","(np.float64(0.7518495631129969), 0.61149110807...","{'Feature': ['tenure', 'monthlycharges', 'tota...","{0: {'Feature': ['tenure', 'monthlycharges', '...","[{'criterion': 'entropy', 'max_depth': 20}, np...","[{'criterion': 'entropy', 'max_depth': 7}, np...."
1,2.34,14,"['paperlessbilling', 'paymentmethod', 'depende...","['totalcharges', 'contract', 'monthlycharges',...","{'ccp_alpha': 0.0, 'class_weight': 'balanced',...","(np.float64(0.7909920475757715), 0.65271006595...","(np.float64(0.7350880092505685), np.float64(0....","(np.float64(0.752938184487951), 0.610666666666...","{'Feature': ['monthlycharges', 'totalcharges',...","{0: {'Feature': ['monthlycharges', 'totalcharg...","[{'criterion': 'gini', 'max_depth': 20}, np.fl...","[{'criterion': 'entropy', 'max_depth': 7}, np...."
2,1.94,6,"['paperlessbilling', 'paymentmethod', 'totalch...","['totalcharges', 'monthlycharges', 'contract',...","{'ccp_alpha': 0.0, 'class_weight': 'balanced',...","(np.float64(0.7982846609092402), 0.66316096990...","(np.float64(0.7357252515707517), np.float64(0....","(np.float64(0.7440738605436892), 0.60302197802...","{'Feature': ['monthlycharges', 'totalcharges',...","{0: {'Feature': ['monthlycharges', 'totalcharg...","[{'criterion': 'gini', 'max_depth': 20}, np.fl...","[{'criterion': 'gini', 'max_depth': 7}, np.flo..."


In [ ]:
df_phone = pd.read_csv('/content/Telco.csv')

df = df_phone.drop('customerID', axis = 1)

# #Check for empty rows

df.isnull().sum()

# #Remove empty rows

df.dropna(inplace = True)

# #Split into X and y

y = df['Churn']
X = df.drop('Churn', axis = 1)

X.columns = [x.lower().strip(' ') for x in X.columns.to_list()]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify = y, random_state = 42)

from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()

categorical_columns = ['gender', 'partner', 'dependents', 'phoneservice', 'multiplelines', 'internetservice',
                       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv',
                       'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'seniorcitizen']


encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoder.fit(X_train[categorical_columns])

X_train_encoded = encoder.transform(X_train[categorical_columns])
X_test_encoded = encoder.transform(X_test[categorical_columns])

encoded_cols = encoder.get_feature_names_out(categorical_columns)

X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoded_cols, index=X_train.index)
X_test_encoded_df  = pd.DataFrame(X_test_encoded,  columns=encoded_cols, index=X_test.index)

X_train = X_train.drop(columns=categorical_columns).join(X_train_encoded_df)
X_test  = X_test.drop(columns=categorical_columns).join(X_test_encoded_df)

#Convert total charges to numeric
X_train['totalcharges'] = pd.to_numeric(X_train['totalcharges'], errors = 'coerce')
X_test['totalcharges'] = pd.to_numeric(X_test['totalcharges'], errors = 'coerce')

X_train = X_train.reset_index(drop=True)
y_train = pd.Series(y_train).reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = pd.Series(y_test).reset_index(drop=True)


X_test_explanations = X_test[:5]

In [ ]:
#The Code below was generated through the use of a large language model and subsequently refined by the authors
import re
import ast

def convert_to_dict(s):
    # 1. Remove dtype specifications
    s = re.sub(r',\s*dtype=[^)]*', '', s)

    # 2. Replace NumPy arrays with Python lists
    # Handles multiline arrays safely
    s = re.sub(
        r'array\((\[[\s\S]*?\])\)',
        r'\1',
        s
    )

    # 3. Replace NumPy scalar types (np.int64, np.float64, etc.)
    s = re.sub(
        r'np\.\w+\(([-+]?\d*\.?\d+(?:e[-+]?\d+)?)\)',
        r'\1',
        s
    )

    # 4. Replace nan / inf if ever present
    s = re.sub(r'\bnan\b', 'None', s, flags=re.IGNORECASE)
    s = re.sub(r'\binf\b', 'None', s, flags=re.IGNORECASE)

    return ast.literal_eval(s)



for x in range(len(df_results_phone_selected)):
  dict_responses = convert_to_dict(df_results_phone_selected['Local Explanations'][x])

  for y in range(5):
    instance = dict_responses[y]
    print(instance)

    ins = X_test_explanations.iloc[y]


    # ---------- INPUTS ----------
    # instance: explanation dict
    # ins: single-row Series from X_test_explanations
    # X_train: training dataframe (needed for averages)

    # ---------- STEP 1: BUILD CONTRIBUTION TABLE ----------
    df_exp = pd.DataFrame({
        'feature': instance['Feature'],
        'weight': instance['Weights'],
        'value': [ins[f] for f in instance['Feature']]
    })

    df_exp['contribution'] = df_exp['weight'] * df_exp['value']


    # ---------- STEP 2: EXTRACT ONE-HOT CATEGORICAL FEATURES ----------
    CATEGORICAL_PREFIXES = {
        'contract': 'contract',
        'payment method': 'paymentmethod',
        'streaming TV': 'streamingtv'
    }

    categorical_rows = []

    for label, prefix in CATEGORICAL_PREFIXES.items():
        subset = df_exp[df_exp['feature'].str.startswith(prefix + '_')]
        active = subset[subset['value'] == 1]

        if not active.empty:
            row = active.iloc[0]
            categorical_rows.append({
                'feature': label,
                'value': row['feature'].split(prefix + '_')[1],
                'contribution': row['contribution']
            })


    # ---------- STEP 3: NUMERIC FEATURES ----------
    NUMERIC_FEATURES = ['monthlycharges', 'totalcharges', 'tenure']

    numeric_rows = []

    for f in NUMERIC_FEATURES:
        if f in df_exp['feature'].values:
            value = ins[f]
            avg = X_train[f].mean()
            comparison = "higher than average" if value > avg else "lower than average"

            contribution = df_exp.loc[df_exp['feature'] == f, 'contribution'].values[0]

            numeric_rows.append({
                'feature': f,
                'value': value,
                'comparison': comparison,
                'contribution': contribution
            })


    # ---------- STEP 4: COMBINE AND SORT ----------
    df_final = pd.concat(
        [pd.DataFrame(numeric_rows), pd.DataFrame(categorical_rows)],
        ignore_index=True
    )

    df_final = df_final.reindex(
        df_final['contribution'].abs().sort_values(ascending=False).index
    )


    # ---------- STEP 5: GENERATE NATURAL LANGUAGE EXPLANATION ----------
    positive, negative = [], []

    for _, row in df_final.iterrows():
        increases_risk = row['contribution'] > 0

        # Numeric feature
        if 'comparison' in row and not pd.isna(row['comparison']):
            sentence = (
                f"their {row['feature']} of {float(row['value']):.2f} "
                f"are {row['comparison']}"
            )

        # Categorical feature
        else:
            sentence = f"and they are on a {row['value']} {row['feature']}"

        if increases_risk:
            positive.append(sentence)
        else:
            negative.append(sentence)


   # ---------- STEP 6: FINAL NARRATIVE (CLASS-AWARE) ----------
    predicted_class = instance['Predicted Class']

    if predicted_class == 1:
        main_outcome = "churn"
        opposite_outcome = "stay"
    else:
        main_outcome = "stay with the company"
        opposite_outcome = "churn"

    if positive:
        explanation = (
            f"The customer is predicted to {main_outcome} because "
            + ", ".join(positive)
        )

        if negative:
            explanation += (
                f", even if "
                + ", ".join(negative)
                + f" would normally suggest they might {opposite_outcome}"
            )

        explanation += "."

    else:
        explanation = (
            f"The customer is predicted to {main_outcome}, even if "
            + ", ".join(negative)
            + "."
        )

    print(explanation)


    print('----------------------------------------------------------')

  print('----------------------------------------------------------')
  print('----------------------------------------------------------')




{'Feature': ['tenure', 'monthlycharges', 'totalcharges', 'contract_Month-to-month', 'contract_One year', 'contract_Two year', 'paymentmethod_Bank transfer (automatic)', 'paymentmethod_Credit card (automatic)', 'paymentmethod_Electronic check', 'paymentmethod_Mailed check'], 'Weights': [-0.04986011, 0.39257778, 0.18756669, -0.11528315, 0.0, -0.01021302, 0.00139374, -0.00051127, 0.01597923, 0.07835012], 'Predicted Class': 0}
The customer is predicted to stay with the company because their totalcharges of 1740.70 are lower than average, their monthlycharges of 96.05 are higher than average, and they are on a Electronic check payment method, even if their tenure of 18.00 are lower than average, and they are on a Month-to-month contract would normally suggest they might churn.
----------------------------------------------------------
{'Feature': ['tenure', 'monthlycharges', 'totalcharges', 'contract_Month-to-month', 'contract_One year', 'contract_Two year', 'paymentmethod_Bank transfer (au

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_y = LabelEncoder()

y_train = le_y.fit_transform(y_train)
y_test = le_y.transform(y_test)

print(y_test[:5])
print(le_y.inverse_transform(y_test[:5]))

[1 0 1 0 0]
['Yes' 'No' 'Yes' 'No' 'No']


#Med Results

In [ ]:
df_results_med = pd.read_csv('/content/results_medical_varied_threshold_full_data_with_idx.csv', index_col = 0)
df_results_med

,Threshold,Number_Columns_Train,Names_Columns_Train,Names_Columns_Final,Final_Model_Parameters,Final_Training_Accuracy,Final_Testing_Accuracy,Global Explanations,Local Explanations,Results_Training_Before_FS,Results_Training_After_FS
0,3.39,24,"['ast', 'relaxation', 'serumcreatinine', 'hemo...","['gender', 'hemoglobin', 'gtp', 'height', 'tri...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.997589,0.789741,"{'Feature': ['gender', 'hemoglobin', 'gtp', 'h...","{0: {'Feature': ['gender', 'hemoglobin', 'gtp'...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 128, 'criterion': 'gini', 'm..."
1,3.34,23,"['ast', 'relaxation', 'serumcreatinine', 'hemo...","['gender', 'hemoglobin', 'gtp', 'height', 'tri...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.997589,0.789741,"{'Feature': ['gender', 'hemoglobin', 'gtp', 'h...","{0: {'Feature': ['gender', 'hemoglobin', 'gtp'...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 256, 'criterion': 'entropy',..."
2,3.19,21,"['ast', 'relaxation', 'serumcreatinine', 'hemo...","['gender', 'hemoglobin', 'gtp', 'height', 'tri...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.997589,0.793333,"{'Feature': ['gender', 'hemoglobin', 'gtp', 'h...","{0: {'Feature': ['gender', 'hemoglobin', 'gtp'...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 256, 'criterion': 'entropy',..."
3,3.09,19,"['ast', 'serumcreatinine', 'hemoglobin', 'gend...","['gender', 'hemoglobin', 'gtp', 'triglyceride'...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.997589,0.789622,"{'Feature': ['gender', 'hemoglobin', 'gtp', 't...","{0: {'Feature': ['gender', 'hemoglobin', 'gtp'...","[{'n_estimators': 256, 'criterion': 'entropy',...","[{'n_estimators': 256, 'criterion': 'entropy',..."
4,3.04,18,"['ast', 'serumcreatinine', 'hemoglobin', 'gend...","['gender', 'hemoglobin', 'triglyceride', 'heig...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999256,0.781003,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{0: {'Feature': ['gender', 'hemoglobin', 'trig...","[{'n_estimators': 256, 'criterion': 'entropy',...","[{'n_estimators': 128, 'criterion': 'entropy',..."
5,2.79,17,"['ast', 'serumcreatinine', 'hemoglobin', 'gend...","['gender', 'hemoglobin', 'triglyceride', 'heig...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999307,0.782918,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{0: {'Feature': ['gender', 'hemoglobin', 'trig...","[{'n_estimators': 256, 'criterion': 'entropy',...","[{'n_estimators': 256, 'criterion': 'entropy',..."
6,2.74,16,"['ast', 'serumcreatinine', 'hemoglobin', 'gend...","['gender', 'hemoglobin', 'triglyceride', 'heig...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999307,0.783218,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{0: {'Feature': ['gender', 'hemoglobin', 'trig...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 256, 'criterion': 'gini', 'm..."
7,2.69,15,"['serumcreatinine', 'hemoglobin', 'gender', 'w...","['gender', 'hemoglobin', 'triglyceride', 'ldl'...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999846,0.780105,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{0: {'Feature': ['gender', 'hemoglobin', 'trig...","[{'n_estimators': 256, 'criterion': 'entropy',...","[{'n_estimators': 128, 'criterion': 'entropy',..."
8,2.64,14,"['serumcreatinine', 'hemoglobin', 'gender', 'w...","['gender', 'hemoglobin', 'triglyceride', 'ldl'...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999872,0.781243,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{0: {'Feature': ['gender', 'hemoglobin', 'trig...","[{'n_estimators': 256, 'criterion': 'entropy',...","[{'n_estimators': 256, 'criterion': 'gini', 'm..."
9,2.59,13,"['hemoglobin', 'gender', 'weight', 'height', '...","['gender', 'hemoglobin', 'triglyceride', 'ldl'...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999872,0.784235,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{

In [ ]:
df_results_med_selected = df_results_med.iloc[[0, 10, 13]]
df_results_med_selected.reset_index(drop=True, inplace=True)
df_results_med_selected

,Threshold,Number_Columns_Train,Names_Columns_Train,Names_Columns_Final,Final_Model_Parameters,Final_Training_Accuracy,Final_Testing_Accuracy,Global Explanations,Local Explanations,Results_Training_Before_FS,Results_Training_After_FS
0,3.39,24,"['ast', 'relaxation', 'serumcreatinine', 'hemo...","['gender', 'hemoglobin', 'gtp', 'height', 'tri...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.997589,0.789741,"{'Feature': ['gender', 'hemoglobin', 'gtp', 'h...","{0: {'Feature': ['gender', 'hemoglobin', 'gtp'...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 128, 'criterion': 'gini', 'm..."
1,2.54,12,"['hemoglobin', 'gender', 'weight', 'height', '...","['gender', 'hemoglobin', 'triglyceride', 'heig...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.999307,0.782799,"{'Feature': ['gender', 'hemoglobin', 'triglyce...","{0: {'Feature': ['gender', 'hemoglobin', 'trig...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 256, 'criterion': 'gini', 'm..."
2,2.34,7,"['hemoglobin', 'gender', 'weight', 'cholestero...","['cholesterol', 'hemoglobin', 'waist', 'gender...","{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.996819,0.770469,"{'Feature': ['cholesterol', 'hemoglobin', 'wai...","{0: {'Feature': ['cholesterol', 'hemoglobin', ...","[{'n_estimators': 256, 'criterion': 'gini', 'm...","[{'n_estimators': 256, 'criterion': 'gini', 'm..."


In [ ]:
#Start with medical dataset
df = pd.read_csv('/content/drive/MyDrive/User Study/smoking.csv')
print(df.shape)

#Data Cleaning

#Drop columns that aren't needed

df = df.drop('ID', axis = 1)

#Check for empty rows

df.isnull().sum()

#Remove empty rows

df.dropna(inplace = True)


#Split into X and y

y = df['smoking']
X = df.drop('smoking', axis = 1)
X.columns = [x.lower().replace(' ', '') for x in X.columns]
X = X.rename(columns={'weight(kg)': 'weight', 'height(cm)': 'height', 'waist(cm)' : 'waist'})

#Remove features with 0 variance

X = X.drop('oral', axis = 1)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify = y, random_state = 42)

X_train = X_train.reset_index(drop=True)
y_train = pd.Series(y_train).reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = pd.Series(y_test).reset_index(drop=True)



# #Convert y column to categorical
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()

# categorical_columns = ['gender', 'dentalcaries', 'tartar']

# for x in categorical_columns:
#   X_train[x] = le.fit_transform(X_train[x])
#   X_test[x] = le.transform(X_test[x])

X_test_explanations = X_test[:5]

(55692, 27)


In [ ]:
#Part of the Code below was generated through the use of a large language model and subsequently refined by the authors
def convert_to_dict(string_dict):
  s2 = re.sub(r'array\(\s*(\[.*?\])\s*\)', r'\1', string_dict)

  # Convert np.int64(x) → x
  s2 = re.sub(r'np\.int64\(\s*([0-9]+)\s*\)', r'\1', s2)


  result = ast.literal_eval(s2)

  return result


for x in range(len(df_results_med_selected)):
  dict_responses = convert_to_dict(df_results_med_selected['Local Explanations'][x])

  for y in range(5):
    instance = dict_responses[y]
    print(instance)

    ins = X_test_explanations.iloc[y]
    print(ins[instance['Feature']])

    print('----------------------------------------------------------')

    # explanation = f"The predicted Class is {instance['Class']} because {instance}"
    # print(explanation)

  print('----------------------------------------------------------')
  print('----------------------------------------------------------')

{'Feature': ['gender', 'hemoglobin', 'gtp', 'height', 'triglyceride'], 'Weights': [0.1192186, 0.03124859, 0.21464748, 0.08479694, 0.04967336], 'Predicted Class': 1}
gender              M
hemoglobin       15.4
gtp             127.0
height            170
triglyceride     89.0
Name: 0, dtype: object
----------------------------------------------------------
{'Feature': ['gender', 'hemoglobin', 'gtp', 'height', 'triglyceride'], 'Weights': [0.09493288, 0.04348911, 0.03922665, 0.0565861, -0.00027477], 'Predicted Class': 1}
gender              M
hemoglobin       16.4
gtp              38.0
height            165
triglyceride    120.0
Name: 1, dtype: object
----------------------------------------------------------
{'Feature': ['gender', 'hemoglobin', 'gtp', 'height', 'triglyceride'], 'Weights': [0.19851153, -0.04378994, 0.03957323, 0.01482293, 0.03348479], 'Predicted Class': 0}
gender             F
hemoglobin      14.8
gtp             18.0
height           160
triglyceride    80.0
Name: 2, dtyp